# YouTube Video Performance & Engagement Analysis
**Using Statistical Analysis and Machine Learning**

**Name:** Mohd Faiz Kashmiri  |  **Section:** B1_B3

**Contents**
1. Setup, data loading & cleaning
2. Feature engineering & descriptive statistics
3. Distributions & outliers
4. Correlation & covariance
5. Probability
6. Sampling techniques & Central Limit Theorem
7. Hypothesis testing (z-test, t-test, ANOVA, proportion test, chi-square)
8. Regression (simple, multiple, log model, diagnostics)
9. K-Nearest Neighbours (classification & regression)
10. Decision trees (classification & regression)
11. Model comparison & conclusions

## 1. Setup, data loading & cleaning

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             r2_score, mean_squared_error, mean_absolute_error)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(42)

In [ ]:
# In Colab: upload youtube_videos_dataset.csv when prompted (skipped if the file is already there)
try:
    from google.colab import files
    if not os.path.exists("youtube_videos_dataset.csv"):
        print("Please upload youtube_videos_dataset.csv")
        files.upload()
except ImportError:
    pass

df = pd.read_csv("youtube_videos_dataset.csv")
display(df.head())
print("Shape:", df.shape)
print(df.dtypes)

In [ ]:
# DATA CLEANING
print("Missing values per column:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

numeric_columns = ["subscribers", "views", "likes", "comments", "duration_minutes", "upload_hour"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["title", "channel", "category", "upload_day"]:
    df[col] = df[col].astype(str).str.strip()

df = df.dropna().reset_index(drop=True)
print("Clean shape:", df.shape)

## 2. Feature engineering & descriptive statistics
New columns used later in tests and models:
* `engagement_rate` = (likes + comments) / views × 100
* `like_rate` = likes / views × 100
* `views_per_subscriber` = views / subscribers
* `is_weekend` – uploaded on Saturday/Sunday
* `time_slot` – Morning / Afternoon / Evening
* `viral` – more than 1,000,000 views
* `high_performer` – views above the median (balanced 0/1 target for classification)

In [ ]:
df["engagement_rate"] = (df["likes"] + df["comments"]) / df["views"] * 100
df["like_rate"] = df["likes"] / df["views"] * 100
df["views_per_subscriber"] = df["views"] / df["subscribers"]
df["is_weekend"] = df["upload_day"].isin(["Saturday", "Sunday"])
df["time_slot"] = pd.cut(df["upload_hour"], bins=[-1, 11, 16, 24], labels=["Morning", "Afternoon", "Evening"])
df["viral"] = (df["views"] > 1_000_000).astype(int)
median_views = df["views"].median()
df["high_performer"] = (df["views"] > median_views).astype(int)

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print("Median views:", median_views)
display(df[["title", "views", "engagement_rate", "views_per_subscriber", "is_weekend", "time_slot", "viral", "high_performer"]].head())

In [ ]:
# DESCRIPTIVE STATISTICS
display(df[numeric_columns].describe().T)

summary = pd.DataFrame({
    "mean": df[numeric_columns].mean(),
    "median": df[numeric_columns].median(),
    "mode": df[numeric_columns].mode().iloc[0],
    "variance": df[numeric_columns].var(),
    "std": df[numeric_columns].std(),
    "IQR": df[numeric_columns].quantile(0.75) - df[numeric_columns].quantile(0.25),
    "CV %": df[numeric_columns].std() / df[numeric_columns].mean() * 100,
    "skewness": df[numeric_columns].skew(),
    "kurtosis": df[numeric_columns].kurt(),
})
display(summary.round(2))

print("\nViews by category")
display(df.groupby("category")["views"].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False).round(0))

## 3. Distributions & outliers

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["views"], bins=15, kde=True, ax=ax[0])
ax[0].set_title("Distribution of YouTube Views (raw)")
ax[0].set_xlabel("Views"); ax[0].set_ylabel("Number of Videos")
sns.histplot(np.log10(df["views"]), bins=15, kde=True, ax=ax[1])
ax[1].set_title("Distribution of log10(Views)")
ax[1].set_xlabel("log10(Views)")
plt.tight_layout(); plt.show()
print("Skewness raw views:", round(df["views"].skew(), 3), "| log views:", round(np.log(df["views"]).skew(), 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].boxplot(df["views"]); ax[0].set_title("Box Plot of Video Views"); ax[0].set_ylabel("Views")
order = df.groupby("category")["views"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="category", y="views", order=order, ax=ax[1])
ax[1].set_title("Views by Category"); ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

# IQR rule for outliers
q1, q3 = df["views"].quantile([0.25, 0.75]); iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["views"] < lo) | (df["views"] > hi)]
print(f"IQR fences: [{lo:,.0f}, {hi:,.0f}] -> {len(outliers)} outlier videos")
display(outliers[["title", "channel", "category", "views"]])

## 4. Correlation & covariance
* **Covariance** measures how two variables move together, but its size depends on the units.
* **Correlation** is covariance divided by both standard deviations, so it always lies between −1 and +1.

In [ ]:
corr_cols = numeric_columns
corr = df[corr_cols].corr()
cov = df[corr_cols].cov()

print("Pearson correlation matrix"); display(corr.round(3))
print("Covariance matrix"); display(cov.round(2))

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=ax[0]); ax[0].set_title("Pearson Correlation")
sns.heatmap(df[corr_cols].corr(method="spearman"), annot=True, cmap="coolwarm", fmt=".2f", ax=ax[1]); ax[1].set_title("Spearman (rank) Correlation")
plt.tight_layout(); plt.show()

In [ ]:
# Covariance and correlation by hand, to show they match the library
x, y = df["subscribers"], df["views"]
n = len(df)
cov_manual = ((x - x.mean()) * (y - y.mean())).sum() / (n - 1)
r_manual = cov_manual / (x.std() * y.std())
print(f"Covariance (manual) : {cov_manual:,.2f}")
print(f"Covariance (numpy)  : {np.cov(x, y)[0, 1]:,.2f}")
print(f"Correlation (manual): {r_manual:.4f}")
print(f"Correlation (pandas): {x.corr(y):.4f}")

# Is each correlation with views statistically significant?
rows = []
for c in corr_cols:
    if c == "views":
        continue
    r, p = stats.pearsonr(df[c], df["views"])
    rho, p_s = stats.spearmanr(df[c], df["views"])
    rows.append({"variable": c, "pearson_r": r, "pearson_p": p, "spearman_rho": rho, "spearman_p": p_s})
display(pd.DataFrame(rows).round(4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=df, x="subscribers", y="views", hue="category", ax=ax[0])
sns.regplot(data=df, x="subscribers", y="views", scatter=False, color="black", ax=ax[0])
ax[0].set_title("Views vs Channel Subscribers")
sns.scatterplot(data=df, x="duration_minutes", y="views", hue="category", legend=False, ax=ax[1])
ax[1].set_title("Views vs Video Duration")
plt.tight_layout(); plt.show()

## 5. Probability
Empirical probability, normal vs log-normal models, conditional probability, Bayes' theorem, addition rule, independence and the binomial distribution.

In [ ]:
views = df["views"]
X = 1_000_000

# Empirical probability
count = (views > X).sum()
p_emp = count / len(df)
print(f"Videos with more than {X:,} views: {count} of {len(df)}")
print(f"Empirical P(views > {X:,}) = {p_emp:.4f} ({p_emp*100:.2f}%)")

# Normal model
mu, sd = views.mean(), views.std()
p_norm = 1 - stats.norm.cdf(X, loc=mu, scale=sd)

# Log-normal model (better for right-skewed data)
shape, loc, scale = stats.lognorm.fit(views, floc=0)
p_logn = 1 - stats.lognorm.cdf(X, shape, loc, scale)

display(pd.DataFrame({"model": ["Empirical", "Normal", "Log-normal"],
                      "P(views > 1M)": [p_emp, p_norm, p_logn]}).round(4))
print("Views are right-skewed, so the normal model usually overestimates the chance of exceeding the mean.")

# Normality tests
print("\nShapiro-Wilk on views     : p =", round(stats.shapiro(views)[1], 6))
print("Shapiro-Wilk on log(views): p =", round(stats.shapiro(np.log(views))[1], 6))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(views, bins=15, density=True, alpha=0.5, label="Observed")
xs = np.linspace(views.min(), views.max(), 300)
ax.plot(xs, stats.norm.pdf(xs, mu, sd), label="Normal fit")
ax.plot(xs, stats.lognorm.pdf(xs, shape, loc, scale), label="Log-normal fit")
ax.axvline(X, color="red", linestyle="--", label="1M views")
ax.set_xlabel("Views"); ax.set_title("Observed views vs fitted models"); ax.legend()
plt.show()

# Empirical (68-95-99.7) rule
for k in [1, 2, 3]:
    within = ((views > mu - k*sd) & (views < mu + k*sd)).mean()
    theory = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"Within {k} SD: observed {within:.3f} vs normal theory {theory:.3f}")

# Z-score of the most viewed video
top = df.loc[views.idxmax()]
print(f"\nMost viewed: '{top['title']}' z-score = {(top['views'] - mu) / sd:.2f}")

In [ ]:
# JOINT, MARGINAL & CONDITIONAL PROBABILITY
print("Joint probability table: category x viral"); display(pd.crosstab(df["category"], df["viral"], normalize="all").round(4))

p_viral = df["viral"].mean()
p_cat = df["category"].value_counts(normalize=True)
p_viral_given_cat = df.groupby("category")["viral"].mean()
print("P(viral | category)"); display(p_viral_given_cat.sort_values(ascending=False).round(4))

# BAYES: P(category | viral) = P(viral | category) * P(category) / P(viral)
if p_viral > 0:
    bayes = (p_viral_given_cat * p_cat / p_viral).sort_values(ascending=False)
    direct = df[df["viral"] == 1]["category"].value_counts(normalize=True)
    display(pd.DataFrame({"Bayes P(cat | viral)": bayes, "Direct count": direct}).fillna(0).round(4))

# ADDITION RULE and INDEPENDENCE (weekend vs viral)
A = df["is_weekend"].astype(bool); B = df["viral"].astype(bool)
pA, pB, pAB = A.mean(), B.mean(), (A & B).mean()
print(f"P(weekend)={pA:.3f}  P(viral)={pB:.3f}  P(both)={pAB:.3f}")
print(f"Addition rule P(weekend OR viral) = {pA + pB - pAB:.3f} (direct: {(A | B).mean():.3f})")
print(f"Independence check: P(A)P(B) = {pA*pB:.3f} vs P(A and B) = {pAB:.3f}")

In [ ]:
# BINOMIAL: if we pick n videos at random, how many are viral?
n_pick = 10
k = np.arange(0, n_pick + 1)
pmf = stats.binom.pmf(k, n_pick, p_viral)
print(f"P(at least 3 viral in {n_pick}) = {1 - stats.binom.cdf(2, n_pick, p_viral):.4f}")
print(f"Expected viral = {n_pick * p_viral:.2f}, SD = {np.sqrt(n_pick * p_viral * (1 - p_viral)):.2f}")
plt.bar(k, pmf); plt.xlabel("Number of viral videos"); plt.ylabel("Probability")
plt.title(f"Binomial(n={n_pick}, p={p_viral:.2f})"); plt.show()

# UNIFORM DISTRIBUTION (from the original notebook)
ud = np.random.uniform(0, 10, 1000)
print("Uniform mean:", ud.mean(), "(theory 5) | variance:", ud.var(), "(theory 8.33)")
plt.hist(ud, bins=20); plt.title("Uniform Distribution"); plt.show()

## 6. Sampling techniques & Central Limit Theorem
Simple random, stratified and systematic sampling; sampling distribution of the mean; standard error; confidence intervals.

In [ ]:
pop_mean = df["views"].mean()
pop_std = df["views"].std(ddof=0)   # population standard deviation
n_s = 30

# 1) Simple random sample
simple = df.sample(n=n_s, random_state=42)

# 2) Stratified sample (same share from every category)
strat = df.groupby("category").sample(frac=n_s / len(df), random_state=42)

# 3) Systematic sample (every k-th row from a random start)
step = max(len(df) // n_s, 1)
start = np.random.RandomState(42).randint(step)
systematic = df.iloc[start::step]

comp = pd.DataFrame({
    "method": ["Population", "Simple random", "Stratified", "Systematic"],
    "n": [len(df), len(simple), len(strat), len(systematic)],
    "mean views": [pop_mean, simple["views"].mean(), strat["views"].mean(), systematic["views"].mean()],
})
comp["error vs population"] = comp["mean views"] - pop_mean
display(comp.round(0))
print("Random sample of videos:")
display(simple[["video_id", "title", "views"]])

In [ ]:
# SAMPLING DISTRIBUTION OF THE MEAN (Central Limit Theorem)
sample_sizes = [10, 30, 50, 100]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, n in zip(axes.ravel(), sample_sizes):
    means = [df["views"].sample(n=n, replace=True).mean() for _ in range(1000)]
    ax.hist(means, bins=30, density=True, alpha=0.6)
    xs = np.linspace(min(means), max(means), 200)
    ax.plot(xs, stats.norm.pdf(xs, pop_mean, pop_std / np.sqrt(n)), color="red")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("Sample mean views")
    print(f"n={n:>3}: mean of means = {np.mean(means):>12,.0f} | empirical SE = {np.std(means):>10,.0f} | theoretical SE = {pop_std/np.sqrt(n):>10,.0f}")
plt.suptitle("Sampling distribution of mean views (red = normal predicted by CLT)")
plt.tight_layout(); plt.show()

In [ ]:
# CONFIDENCE INTERVALS (95%)
m, s = simple["views"].mean(), simple["views"].std()
t_lo, t_hi = stats.t.interval(0.95, df=n_s - 1, loc=m, scale=stats.sem(simple["views"]))
z_lo, z_hi = stats.norm.interval(0.95, loc=m, scale=pop_std / np.sqrt(n_s))
print(f"Sample mean = {m:,.0f}")
print(f"95% t-interval (sigma unknown): [{t_lo:,.0f}, {t_hi:,.0f}]")
print(f"95% z-interval (sigma known)  : [{z_lo:,.0f}, {z_hi:,.0f}]")
print(f"Population mean = {pop_mean:,.0f}")

# Coverage simulation: about 95% of intervals should contain the true mean
hits = 0
for _ in range(1000):
    smp = df["views"].sample(n=n_s, replace=True)
    lo_, hi_ = stats.t.interval(0.95, df=n_s - 1, loc=smp.mean(), scale=stats.sem(smp))
    hits += (lo_ <= pop_mean <= hi_)
print(f"Coverage of 1000 simulated 95% intervals: {hits / 10:.1f}%")

## 7. Hypothesis testing
General procedure: state H0 and H1, choose α = 0.05, compute the test statistic and p-value, and compare. Reject H0 when p < α.

In [ ]:
ALPHA = 0.05
results = []

def conclude(name, h0, stat, p):
    decision = "Reject H0" if p < ALPHA else "Fail to reject H0"
    results.append({"test": name, "H0": h0, "statistic": round(float(stat), 4), "p-value": round(float(p), 4), "decision": decision})
    print(f"{name}\n   statistic = {stat:.4f} | p-value = {p:.4f} -> {decision} at alpha = {ALPHA}\n")

views = df["views"]
mu0 = 1_000_000

### 7.1 One-sample z-test
H0: mean views = 1,000,000 vs H1: mean views ≠ 1,000,000. The z-test assumes σ is known; here the population standard deviation of the dataset is used.

In [ ]:
n = len(views)
sigma = views.std(ddof=0)
z = (views.mean() - mu0) / (sigma / np.sqrt(n))
p_two = 2 * (1 - stats.norm.cdf(abs(z)))
p_right = 1 - stats.norm.cdf(z)
print(f"z = {z:.4f}, critical values = ±{stats.norm.ppf(1 - ALPHA/2):.3f}")
print(f"Right-tailed p-value (H1: mean > 1M) = {p_right:.4f}")
conclude("One-sample z-test (mean views = 1M)", "mu = 1,000,000", z, p_two)

### 7.2 One-sample t-test
Same hypothesis, but σ is estimated from the sample, which is the realistic situation.

In [ ]:
t_stat, p_t = stats.ttest_1samp(views, mu0)
ci = stats.t.interval(0.95, df=n - 1, loc=views.mean(), scale=stats.sem(views))
print(f"95% CI for mean views: [{ci[0]:,.0f}, {ci[1]:,.0f}]")
conclude("One-sample t-test (mean views = 1M)", "mu = 1,000,000", t_stat, p_t)

### 7.3 Two-sample tests: weekend vs weekday uploads
H0: mean views are equal for weekend and weekday uploads. Welch's t-test does not assume equal variances. The two-sample z-test and Mann-Whitney U test are shown for comparison.

In [ ]:
g_we = df.loc[df["is_weekend"], "views"]
g_wd = df.loc[~df["is_weekend"], "views"]
print(f"Weekend n={len(g_we)}, mean={g_we.mean():,.0f} | Weekday n={len(g_wd)}, mean={g_wd.mean():,.0f}")

# Equal variance check
print("Levene test for equal variances: p =", round(stats.levene(g_we, g_wd)[1], 4))

# Welch t-test
t2, p2 = stats.ttest_ind(g_we, g_wd, equal_var=False)
conclude("Welch two-sample t-test (weekend vs weekday)", "mu_weekend = mu_weekday", t2, p2)

# Two-sample z-test (large-sample approximation)
z2 = (g_we.mean() - g_wd.mean()) / np.sqrt(g_we.var()/len(g_we) + g_wd.var()/len(g_wd))
conclude("Two-sample z-test (weekend vs weekday)", "mu_weekend = mu_weekday", z2, 2 * (1 - stats.norm.cdf(abs(z2))))

# Non-parametric alternative
u, pu = stats.mannwhitneyu(g_we, g_wd, alternative="two-sided")
conclude("Mann-Whitney U (weekend vs weekday)", "same distribution", u, pu)

# Effect size (Cohen's d)
pooled = np.sqrt(((len(g_we)-1)*g_we.var() + (len(g_wd)-1)*g_wd.var()) / (len(g_we)+len(g_wd)-2))
print("Cohen's d =", round((g_we.mean() - g_wd.mean()) / pooled, 3))

### 7.4 Two-sample t-test: Education vs Science & Technology

In [ ]:
a = df.loc[df["category"] == "Education", "views"]
b = df.loc[df["category"] == "Science & Technology", "views"]
print(f"Education n={len(a)}, mean={a.mean():,.0f} | Science & Technology n={len(b)}, mean={b.mean():,.0f}")
t3, p3 = stats.ttest_ind(a, b, equal_var=False)
conclude("Welch t-test (Education vs Science & Technology)", "mu_edu = mu_sci", t3, p3)

### 7.5 One-way ANOVA: do mean views differ across categories?

In [ ]:
groups = [g["views"].values for _, g in df.groupby("category") if len(g) >= 2]
f, p_f = stats.f_oneway(*groups)
conclude("One-way ANOVA (views across categories)", "all category means equal", f, p_f)
h, p_h = stats.kruskal(*groups)
conclude("Kruskal-Wallis (non-parametric ANOVA)", "all category distributions equal", h, p_h)

# ANOVA on log(views) is safer because raw views are skewed
f_l, p_l = stats.f_oneway(*[np.log(g) for g in groups])
conclude("One-way ANOVA on log(views)", "all category means equal", f_l, p_l)

### 7.6 Two-proportion z-test: is a weekend video more likely to go viral?
H0: p(viral | weekend) = p(viral | weekday).

In [ ]:
x1, n1 = df.loc[df["is_weekend"], "viral"].sum(), df["is_weekend"].sum()
x2, n2 = df.loc[~df["is_weekend"], "viral"].sum(), (~df["is_weekend"]).sum()
p1, p2_ = x1 / n1, x2 / n2
p_pool = (x1 + x2) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
zp = (p1 - p2_) / se if se > 0 else 0.0
print(f"Weekend viral share = {p1:.3f} ({x1}/{n1}) | Weekday = {p2_:.3f} ({x2}/{n2})")
conclude("Two-proportion z-test (viral: weekend vs weekday)", "p_weekend = p_weekday", zp, 2 * (1 - stats.norm.cdf(abs(zp))))

### 7.7 Chi-square tests
**Test of independence:** are two categorical variables related? **Goodness of fit:** do uploads follow a given distribution?

In [ ]:
def chi_square_test(a, b, label):
    ct = pd.crosstab(a, b)
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n_ = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n_ * (min(ct.shape) - 1)))
    print("Observed counts:"); display(ct)
    low = (expected < 5).mean() * 100
    print(f"dof = {dof}, Cramer's V = {cramers_v:.3f}, cells with expected count < 5: {low:.0f}%")
    if low > 20:
        print("Warning: many small expected counts, so the chi-square approximation is unreliable. Treat as indicative.")
    conclude(label, "variables are independent", chi2, p)

chi_square_test(df["category"], df["high_performer"], "Chi-square independence (category vs high_performer)")
chi_square_test(df["time_slot"], df["high_performer"], "Chi-square independence (time_slot vs high_performer)")
chi_square_test(df["is_weekend"], df["viral"], "Chi-square independence (weekend vs viral)")

In [ ]:
# Goodness of fit: are uploads spread evenly over the 7 days?
observed = df["upload_day"].value_counts().reindex(DAYS).fillna(0)
chi2_gof, p_gof = stats.chisquare(observed)   # expected = equal counts
print("Expected per day:", round(len(df) / 7, 2))
display(observed.to_frame("observed"))
conclude("Chi-square goodness of fit (uploads uniform over days)", "uploads equally likely each day", chi2_gof, p_gof)

# Normality test
sh, p_sh = stats.shapiro(df["views"])
conclude("Shapiro-Wilk normality (views)", "views are normal", sh, p_sh)

In [ ]:
print("SUMMARY OF ALL HYPOTHESIS TESTS")
display(pd.DataFrame(results))

## 8. Regression
We separate **honest predictors** (subscribers, duration, upload time, category) from **likes and comments**, which are almost perfectly correlated with views (r ≈ 0.99) and would leak the answer.

In [ ]:
def make_features(d, include_engagement=False):
    X_ = d[["subscribers", "duration_minutes", "upload_hour"]].copy()
    X_["is_weekend"] = d["is_weekend"].astype(int)
    X_ = pd.concat([X_, pd.get_dummies(d["category"], prefix="cat", drop_first=True, dtype=int)], axis=1)
    if include_engagement:
        X_["likes"] = d["likes"]; X_["comments"] = d["comments"]
    return X_

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

### 8.1 Simple linear regression: views ~ subscribers

In [ ]:
res = stats.linregress(df["subscribers"], df["views"])
print(f"views = {res.intercept:,.0f} + {res.slope:.3f} x subscribers")
print(f"R^2 = {res.rvalue**2:.4f} | r = {res.rvalue:.4f} | slope p-value = {res.pvalue:.2e} | slope SE = {res.stderr:.4f}")
print(f"Interpretation: every extra 1,000 subscribers is associated with about {res.slope*1000:,.0f} more views.")
print(f"Predicted views for a 200,000-subscriber channel: {res.intercept + res.slope*200_000:,.0f}")

fitted = res.intercept + res.slope * df["subscribers"]
resid = df["views"] - fitted
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(df["subscribers"], df["views"]); ax[0].plot(df["subscribers"].sort_values(), res.intercept + res.slope*df["subscribers"].sort_values(), color="red")
ax[0].set_xlabel("Subscribers"); ax[0].set_ylabel("Views"); ax[0].set_title("Simple linear regression")
ax[1].scatter(fitted, resid); ax[1].axhline(0, color="red", linestyle="--")
ax[1].set_xlabel("Fitted values"); ax[1].set_ylabel("Residuals"); ax[1].set_title("Residuals vs fitted")
plt.tight_layout(); plt.show()

### 8.2 Multiple linear regression

In [ ]:
X_all = make_features(df)
y_all = df["views"]
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

lin = LinearRegression().fit(X_tr, y_tr)
pred = lin.predict(X_te)
n_tr, p_tr = X_tr.shape
r2_tr = lin.score(X_tr, y_tr)
print(f"Train R^2 = {r2_tr:.3f} | Adjusted R^2 = {1 - (1 - r2_tr) * (n_tr - 1) / (n_tr - p_tr - 1):.3f}")
print(f"Test  R^2 = {r2_score(y_te, pred):.3f} | RMSE = {rmse(y_te, pred):,.0f} | MAE = {mean_absolute_error(y_te, pred):,.0f}")

cv = cross_val_score(LinearRegression(), X_all, y_all, cv=KFold(5, shuffle=True, random_state=42), scoring="r2")
print(f"5-fold CV R^2 = {cv.mean():.3f} (+/- {cv.std():.3f})")

display(pd.DataFrame({"feature": X_all.columns, "coefficient": lin.coef_}).sort_values("coefficient", key=abs, ascending=False).round(3))
plt.scatter(y_te, pred); lims = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
plt.plot(lims, lims, "r--"); plt.xlabel("Actual views"); plt.ylabel("Predicted views"); plt.title("Actual vs predicted (test set)"); plt.show()

### 8.3 Log model, leakage and multicollinearity

In [ ]:
# Log-transform the target to handle skew
log_model = LinearRegression().fit(X_tr, np.log(y_tr))
pred_log = np.exp(log_model.predict(X_te))
print(f"Log model: test R^2 (original scale) = {r2_score(y_te, pred_log):.3f} | RMSE = {rmse(y_te, pred_log):,.0f}")

# Adding likes and comments inflates R^2 (leakage)
X_leak = make_features(df, include_engagement=True)
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak, y_all, test_size=0.2, random_state=42)
leak = LinearRegression().fit(Xl_tr, yl_tr)
print(f"With likes & comments: test R^2 = {leak.score(Xl_te, yl_te):.3f}  <- looks great, but likes/comments are consequences of views")

# Variance Inflation Factor
def vif(frame):
    out = {}
    for c in frame.columns:
        r2 = LinearRegression().fit(frame.drop(columns=c), frame[c]).score(frame.drop(columns=c), frame[c])
        out[c] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(out)
display(vif(df[["subscribers", "likes", "comments", "duration_minutes", "upload_hour"]]).round(1).to_frame("VIF"))
print("VIF above 10 signals strong multicollinearity (likes and comments overlap heavily).")

# Residual diagnostics
res_ = y_te - pred
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
stats.probplot(res_, dist="norm", plot=ax[0]); ax[0].set_title("Q-Q plot of residuals")
sns.histplot(res_, kde=True, ax=ax[1]); ax[1].set_title("Residual distribution")
plt.tight_layout(); plt.show()

## 9. K-Nearest Neighbours
Target for classification: `high_performer` (views above the median). KNN is distance-based, so features are standardised.

In [ ]:
X_cls = make_features(df)
y_cls = df["high_performer"]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

# Choose k by 5-fold cross-validation on the training data
skf = StratifiedKFold(5, shuffle=True, random_state=42)
ks = range(1, 16)
cv_acc = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), Xc_tr, yc_tr, cv=skf).mean() for k in ks]
best_k = list(ks)[int(np.argmax(cv_acc))]
plt.plot(list(ks), cv_acc, marker="o"); plt.axvline(best_k, color="red", linestyle="--")
plt.xlabel("k"); plt.ylabel("CV accuracy"); plt.title(f"Choosing k (best k = {best_k})"); plt.show()

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(best_k)).fit(Xc_tr, yc_tr)
knn_pred = knn.predict(Xc_te)
print(f"KNN (k={best_k}) test accuracy = {accuracy_score(yc_te, knn_pred):.3f}")
print(classification_report(yc_te, knn_pred, target_names=["Below median", "Above median"], zero_division=0))
sns.heatmap(confusion_matrix(yc_te, knn_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Below", "Above"], yticklabels=["Below", "Above"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("KNN confusion matrix"); plt.show()

In [ ]:
# KNN regression: predict the number of views
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_cls, df["views"], test_size=0.2, random_state=42)
kf = KFold(5, shuffle=True, random_state=42)
cv_r2 = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsRegressor(k)), Xr_tr, yr_tr, cv=kf, scoring="r2").mean() for k in ks]
best_kr = list(ks)[int(np.argmax(cv_r2))]
knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(best_kr)).fit(Xr_tr, yr_tr)
pr = knn_reg.predict(Xr_te)
print(f"KNN regressor (k={best_kr}): test R^2 = {r2_score(yr_te, pr):.3f} | RMSE = {rmse(yr_te, pr):,.0f}")

## 10. Decision trees

In [ ]:
param_grid = {"max_depth": [2, 3, 4, 5, 6], "min_samples_leaf": [1, 2, 3, 5]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=skf, scoring="accuracy").fit(Xc_tr, yc_tr)
tree = grid.best_estimator_
print("Best parameters:", grid.best_params_, "| CV accuracy:", round(grid.best_score_, 3))

tree_pred = tree.predict(Xc_te)
print(f"Decision tree test accuracy = {accuracy_score(yc_te, tree_pred):.3f}")
print(classification_report(yc_te, tree_pred, target_names=["Below median", "Above median"], zero_division=0))

plt.figure(figsize=(20, 9))
plot_tree(tree, feature_names=list(Xc_tr.columns), class_names=["Below median", "Above median"], filled=True, rounded=True, fontsize=9)
plt.title("Decision tree for high_performer"); plt.show()
print(export_text(tree, feature_names=list(Xc_tr.columns)))

In [ ]:
imp = pd.Series(tree.feature_importances_, index=Xc_tr.columns).sort_values()
imp[imp > 0].plot.barh(figsize=(8, 5)); plt.title("Decision tree feature importance"); plt.show()

sns.heatmap(confusion_matrix(yc_te, tree_pred), annot=True, fmt="d", cmap="Greens",
            xticklabels=["Below", "Above"], yticklabels=["Below", "Above"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Decision tree confusion matrix"); plt.show()

# Decision tree regression
gr = GridSearchCV(DecisionTreeRegressor(random_state=42), param_grid, cv=kf, scoring="r2").fit(Xr_tr, yr_tr)
tr = gr.best_estimator_.predict(Xr_te)
print("Tree regressor best params:", gr.best_params_)
print(f"Tree regressor: test R^2 = {r2_score(yr_te, tr):.3f} | RMSE = {rmse(yr_te, tr):,.0f}")

## 11. Model comparison & conclusions

In [ ]:
models = {
    "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    f"KNN (k={best_k})": make_pipeline(StandardScaler(), KNeighborsClassifier(best_k)),
    f"Decision tree {grid.best_params_}": DecisionTreeClassifier(random_state=42, **grid.best_params_),
}
skf10 = StratifiedKFold(5, shuffle=True, random_state=7)
rows = []
for name, m in models.items():
    s = cross_val_score(m, X_cls, y_cls, cv=skf10, scoring="accuracy")
    rows.append({"model": name, "mean accuracy": s.mean(), "std": s.std()})
comparison = pd.DataFrame(rows).sort_values("mean accuracy", ascending=False)
display(comparison.round(3))
plt.barh(comparison["model"], comparison["mean accuracy"], xerr=comparison["std"])
plt.xlabel("5-fold CV accuracy"); plt.title("Classifier comparison (high_performer)"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

### Key findings
* **Distribution:** views are strongly right-skewed with a few very large outliers, so the normal model overstates the probability of exceeding 1M views (about 51% under the normal model versus 32.7% observed).
* **Correlation:** views, likes and comments move almost together (r ≈ 0.99); subscribers are strongly related to all three (r ≈ 0.83); video duration has almost no relationship with performance (r ≈ 0.05).
* **Sampling:** the sampling distribution of the mean becomes narrower and closer to normal as the sample size grows (Central Limit Theorem), and 95% confidence intervals cover the true mean about 95% of the time.
* **Hypothesis tests:** see the summary table in section 7 for each decision at α = 0.05.
* **Regression:** subscriber count is the dominant predictor of views. Adding likes and comments inflates R² only because they are consequences of views, not causes.
* **Machine learning:** KNN and decision trees are compared against a baseline using cross-validation; with only 55 videos, results vary between splits, so more data would make the conclusions firmer.
* **Limitations:** small sample (n = 55), correlation is not causation, and categories with few videos give unstable estimates.